# LDA TOPIC MODELING


## This notebook applies LDA modeling to a an experimental dataset investigating participants' goal inferences

## The utility of topic modeling methods is their capability to uncover unobserved variables—topics—which shape the meaning of textual documents. 

### In the following case, participants’ read a "screen grab" of a news story covering sexual assault. We asked participants what they thought the author was trying to do by writing the news story. Then, we instructed them to list up to 10 inferences about the author's goals were in writing the news report (e.g., "to write an interesting story").


using the GENSIM, NLTK, spaCy, and SKLearn libraries:

https://radimrehurek.com/gensim/

https://www.nltk.org

https://scikit-learn.org/stable/

https://spacy.io


and folowing a really useful tutorial from the following journal article: 

https://link_to_article_here.com


# Steps

## 1. Preparing the text for preprocessing
    1a. Spell check
    1b. Expand contractions

## 2. Text preprocessing

     2a. Read in data
 
     2b. Tokenization
     
     2c. Stop Word Removal
     
     2d. Stemming
     
     2e. Bigrams and Trigrams
     
     2f. Exclude terms in > 99% and < 1% of documents
     
     2g. Generate Corpus and Dictionary
 

## 2. Selecting the number of topics (k)
 
     2a. Computing Model Perplexity


## 3. Model Results

     3a. pyLDAvis visualization to assist with Topic Labeling
     
     3b. Topic Mixtures (Document-Term Matrix)

 
# Helpful Links:

https://medium.com/@lettier/how-does-lda-work-ill-explain-using-emoji-108abf40fa7d

     
https://towardsdatascience.com/light-on-math-machine-learning-intuitive-guide-to-latent-dirichlet-allocation-437c81220158


# Load Required Libraries

In [1]:
import nltk
import numpy as np
import pandas as pd
from pandas import read_excel

from nltk.corpus import stopwords
from nltk.tag import pos_tag

from sklearn import feature_extraction
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.externals import joblib
from sklearn.manifold import MDS

import spacy
from spacy.lang.en import English

# Compute bigrams.
from gensim.models import Phrases
from gensim.utils import simple_preprocess

from IPython.display import display 
import vpython as vs
import matplotlib as mpl
from matplotlib import pyplot as plt
%matplotlib inline
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
from matplotlib.ticker import FuncFormatter

import mpld3
from mpld3 import plugins, utils
import plotly
import plotly.graph_objs as go
import json
import pyLDAvis
import pyLDAvis.gensim

from sklearn.cluster import KMeans
from sklearn.utils import shuffle

import gensim
from gensim.models.wrappers import LdaMallet
from gensim.test.utils import common_corpus, common_dictionary
from gensim import corpora, models, similarities
from gensim.models.ldamodel import LdaModel
from gensim.models.coherencemodel import CoherenceModel
from PIL import *
import pickle

# ----------------------------------------------Misc---------------------------------------
import re
import csv
import os 
import codecs
import collections as cs
import logging
import random

pd.set_option('display.max_rows', 5000)
pd.set_option('display.max_columns', 5000)
pd.set_option('display.width', 10000)

/Users/hannahstevens/anaconda3/envs/first/lib/python3.6/site-packages/sklearn/externals/joblib/__init__.py:15: DeprecationWarning: sklearn.externals.joblib is deprecated in 0.21 and will be removed in 0.23. Please import this functionality directly from joblib, which can be installed with: pip install joblib. If this warning is raised when loading pickled models, you may need to re-serialize those models with scikit-learn 0.21+.
  warnings.warn(msg, category=DeprecationWarning)


<IPython.core.display.Javascript object>

# 1. Before you read in your data, you should do the following:

## 1a. Run your textual data through a spell checker
     Altough there are automated spell checkers, they aren't as accurate as we'd like. Thus, we 
     encourage you to have a human run it through a spellchecker
## 1b. Expand all english contractions (e.g., "don't" -> "do not")
    Similar to the spellchecker, we needed human coders to do this, to ensure accuracy


# 2. DATA PREPROCESSING

# 2a. Read in the following:
### 1. Experimental dataset
### 2. Stopword location
### 3. Gensim location
### 4. Mallet path

In [2]:
#Paths
file_location = '/Users/hannahstevens/Desktop/LDA_Main/project2/data_new/experimental_data.xlsx'
stopwords_location = '/Users/hannahstevens/Desktop/LDA_Main/project2/data_new/stopwords.txt'
log_location = '/Users/hannahstevens/Desktop/LDA_Main/project2/data_new/gensim.log'
os.environ['MALLET_HOME'] = '/Users/hannahstevens/Desktop/LDA_Main/mallet-2.0.8'
mallet_path = '/Users/hannahstevens/Desktop/LDA_Main/mallet-2.0.8/bin/mallet'

## Did the dataset load with the correct number of columns and rows?

In [3]:
try:
    data = pd.read_excel(file_location, encoding='latin1')
    print("{} Rows.  {} Columns.".format(*data.shape))
except:
    print("Dataset could not be loaded. Is the dataset missing?")

813 Rows.  283 Columns.


In [5]:
# Randomimze the order of the rows in the dataframe
data = shuffle(data)

## Check to make sure the dataset looks correct

In [40]:
try:
    data = pd.read_excel(file_location, encoding='latin1')
    print("{} Rows.  {} Columns.".format(*data.shape))
except:
    print("Dataset could not be loaded. Is the dataset missing?")

813 Rows.  283 Columns.


In [41]:
indices = [0,333,777,932]

samples = pd.DataFrame(data.loc[indices], columns = data.keys()).reset_index(drop = True)
print("Sample Tickets:")
display(samples)

Sample Tickets:


/Users/hannahstevens/anaconda3/envs/first/lib/python3.6/site-packages/ipykernel_launcher.py:3: FutureWarning:


Passing list-likes to .loc or [] with any missing label will raise
KeyError in the future, you can use .reindex() as an alternative.

See the documentation here:
https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#deprecate-loc-reindex-listlike



,uid,All_merged,GI_AM_merged,GI_Femstud_merged,GI_Author_merged,GI_Author_merged_oroginal,StartDate,EndDate,Status,IPAddress,Progress,Duration (in seconds),Finished,RecordedDate,ResponseId,RecipientLastName,RecipientFirstName,RecipientEmail,ExternalReference,LocationLatitude,LocationLongitude,DistributionChannel,UserLanguage,time_introv_First Click,time_introv_Last Click,time_introv_Page Submit,time_introv_Click Count,time_instr_First Click,time_instr_Last Click,time_instr_Page Submit,time_instr_Click Count,timer_01_First Click,timer_01_Last Click,timer_01_Page Submit,timer_01_Click Count,timer_02_First Click,timer_02_Last Click,timer_02_Page Submit,timer_02_Click Count,timer_03_First Click,timer_03_Last Click,timer_03_Page Submit,timer_03_Click Count,timer_04_First Click,timer_04_Last Click,timer_04_Page Submit,timer_04_Click Count,timer_05_First Click,timer_05_Last Click,timer_05_Page Submit,timer_05_Click Count,timer_06_First Click,timer_06_Last Click,timer_06_Page Submit,timer_06_Click Count,timer_07_First Click,timer_07_Last Click,timer_07_Page Submit,timer_07_Click Count,timer_08_First Click,timer_08_Last Click,timer_08_Page Submit,timer_08_Click Count,timer_09_First Click,timer_09_Last Click,timer_09_Page Submit,timer_09_Click Count,timer_10_First Click,timer_10_Last Click,timer_10_Page Submit,timer_10_Click Count,timer_11_First Click,timer_11_Last Click,timer_11_Page Submit,timer_11_Click Count,timer_12_First Click,timer_12_Last Click,timer_12_Page Submit,timer_12_Click Count,timer_13_First Click,timer_13_Last Click,timer_13_Page Submit,timer_13_Click Count,timer_14_First Click,timer_14_Last Click,timer_14_Page Submit,timer_14_Click Count,gi_AM_1,gi_AM_2,gi_AM_3,gi_AM_4,gi_AM_5,gi_AM_6,gi_AM_7,gi_AM_8,gi_AM_9,gi_AM_10,gi_femstud_1,gi_femstud_2,gi_femstud_3,gi_femstud_4,gi_femstud_5,gi_femstud_6,gi_femstud_7,gi_femstud_8,gi_femstud_9,gi_femstud_10,gi_author_1,gi_author_2,gi_author_3,gi_author_4,gi_author_5,gi_author_6,gi_author_7,gi_author_8,gi_author_9,gi_author_10,gi_author_1 gi_author_2 gi_author_3 gi_author_4 gi_author_5 gi_author_6 gi_author_7 gi_author_8 gi_author_9 gi_author_10,blame_femstud_1,blame_femstud_2,blame_femstud_3,blame_femstud_4,blame_femstud_5,blame_femstud_6,blame_AM_1,blame_AM_2,blame_AM_3,blame_AM_4,blame_AM_5,blame_AM_6,relative_blame1_1,relative_blame1_2,relative_blame2_1,relative_blame2_2,relative_blame2_3,rape1_1,rape1_2,rape1_3,rape1_4,rape1_5,rape1_6,rape1_7,rape1_8,rape1_9,rape1_10,Consent_before_1,Consent_before_2,Consent_before_3,Consent_before_4,Consent_before_5,Consent_before_6,Consent_before_7,Consent_before_8,Consent_before_9,Consent_before_10,Consent_before_11,Consent_before_12,Consent_after_1,Consent_after_2,Consent_after_3,Consent_after_4,Consent_after_5,Consent_after_6,Consent_after_7,Consent_after_8,Consent_after_9,Consent_after_10,Consent_after_11,Consent_after_12,REMV_1,REMV_2,REMV_3,REMV_4,REMV_5,REMV_6,REMV_7,REMV_8,REMP_1,REMP_2,REMP_3,REMP_4,REMP_5,REMP_6,REMP_7,REMP_8,status_1,status_2,status_3,status_4,status_5,status_6,status_7,status_8,status_9,status_10,status_11,status_12,status_13,status_14,status_15,status_16,position_recog,quote_recog,alcohol_morris,alcohol_femstudent,alcohol_present,intoxication_AM_1,intoxication_AM_2,intoxication_AM_3,intoxication_femstud_1,intoxication_femstud_2,intoxication_femstud_3,realism_1,realism_2,realism_3,realism_4,realism_5,realism_6,realism_7,realism_8,realism_9,worked_aggie,journalism_exp,partic_sex_assault,age,sex,race,race_7_TEXT,orientation,orientation_5_TEXT,RMA_1,RMA_2,RMA_3,RMA_4,RMA_5,RMA_6,RMA_7,RMA_8,RMA_9,RMA_10,RMA_11,RMA_12,RMA_13,RMA_14,RMA_15,RMA_16,RMA_17,RMA_18,RMA_19,RMA_20,RMA_21,RMA_22,RMA_23,RMA_24,RMA_25,RMA_26,RMA_27,RMA_28,RMA_29,RMA_30,RMA_31,RMA_32,RMA_33,RMA_34,RMA_35,RMA_36,RMA_37,RMA_38,RMA_39,RMA_40,RMA_41,RMA_42,RMA_43,RMA_44,RMA_45,alchohol,status1,status2,topic,gi_author_8 - Sentiment Polarity,gi_author_8 - Sentiment Score,gi_author_8 - Sentiment,gi_author_8 - Topics
0,1.0,NaN,NaN,NaN,Get readers Expos

# 2b. Tokenization
### Tokenization involves coverting the text to lowercase, removing special characters, null valies, and punctuation from the text

## Here we can see how many null values there are in each column

In [42]:
# number of null values in each column of the full dataset
data.isnull().sum()

uid                                                                                                                           1
All_merged                                                                                                                  813
GI_AM_merged                                                                                                                813
GI_Femstud_merged                                                                                                           813
GI_Author_merged                                                                                                              0
GI_Author_merged_oroginal                                                                                                     0
StartDate                                                                                                                     1
EndDate                                                                                                 

## Now we need to remove null values from the data

In [43]:
#finding null values in the full dataset
print("=============Full Dataset=============")
data['GI_Author_merged'] = data['GI_Author_merged']

print('Number of rows in GI_Author_merged:', len(data['GI_Author_merged']))

print("-------------------")
print("Null Values in GI_Author_merged: {}".format(data['GI_Author_merged'].isnull().sum()))

#Removing null values from the full dataset

GI_Author_merged = data['GI_Author_merged']

print("After removing Null Values in Full Dataset")
print("Null Values in GI_Author_merged: {}".format(data['GI_Author_merged'].isnull().sum()))



=============Full Dataset=============
Number of rows in GI_Author_merged: 813
-------------------
Null Values in GI_Author_merged: 0
After removing Null Values in Full Dataset
Null Values in GI_Author_merged: 0


## Convert the text to lowercase

In [44]:
#----------------------------------------Convert everything to Lower case--------------------------------------------------

##Train Data
GI_Author_merged = GI_Author_merged.str.lower()

print("=======Full Dataset==============\n")
print(GI_Author_merged.head(1))


=======Full Dataset==============

0    get readers expose sexual violence report in a...
Name: GI_Author_merged, dtype: object


## Remove the following:
- special characters
- alphanumerics
- numbers
- words that appear in the corpus less than twice
- extra spaces

In [45]:
##Remove special characters from full dataset

GI_Author_merged_regex = [re.sub(r'\S*@\S*\s?', '', sent) for sent in GI_Author_merged]
GI_Author_merged_regex = [re.sub(r'\'', '', sent) for sent in GI_Author_merged_regex]
GI_Author_merged_regex = [re.sub(r'[^\w\s]', '', sent) for sent in GI_Author_merged_regex]
GI_Author_merged_regex = [re.sub(r'\d', '',  sent) for sent in GI_Author_merged_regex]
GI_Author_merged_regex = [re.sub(r'\W*\b\w{1,2}\b', '',  sent) for sent in GI_Author_merged_regex]
GI_Author_merged_regex = [re.sub(r'_', ' ',  sent) for sent in GI_Author_merged_regex]

print("=======Full Dataset==============\n")
print("\n[INFO] GI_Author_merged....................\n")
print(GI_Author_merged_regex[:2])


=======Full Dataset==============


[INFO] GI_Author_merged....................

['get readers expose sexual violence report unbiased manner get interesting story pass class raise awareness report the facts get engagement write about something people care about expose the guy', '         ']


## Remove All Punctuation

In [13]:
def tokenize(sentences):
    for sentence in sentences:
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))  # deacc=True removes all punctuation

In [46]:
## Full Data set
GI_Author_merged_tokens = list(tokenize(GI_Author_merged_regex))


print("\n[INFO] GI_Author_merged....................\n")
print(GI_Author_merged_tokens[:2])


[INFO] GI_Author_merged....................

[['get', 'readers', 'expose', 'sexual', 'violence', 'report', 'unbiased', 'manner', 'get', 'interesting', 'story', 'pass', 'class', 'raise', 'awareness', 'report', 'the', 'facts', 'get', 'engagement', 'write', 'about', 'something', 'people', 'care', 'about', 'expose', 'the', 'guy'], []]


# 2c. Remove Stopwords
## NOTE: Edit the stopwords txt file to add additional words to filter out

In [17]:
#Prepare to remove stopwords
nltk.download('stopwords')
stopwords = set(nltk.corpus.stopwords.words('english'))
newStopWords =[str(x.strip()) for x in open(stopwords_location,'r').read().split('\n')]
stopwords.update(newStopWords)

def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) if word not in stopwords] for doc in texts]
print(len(stopwords))

210


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/hannahstevens/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [47]:
##Remove stopwords 

GI_Author_merged_stopwords = remove_stopwords(GI_Author_merged_tokens)


print("\n[INFO] GI_Author_merged....................\n")
print(GI_Author_merged_stopwords[:2])




[INFO] GI_Author_merged....................

[['get', 'expose', 'sexual', 'violence', 'report', 'unbiased', 'manner', 'get', 'interesting', 'story', 'pass', 'class', 'raise', 'awareness', 'report', 'facts', 'get', 'engagement', 'write', 'something', 'people', 'care', 'expose', 'guy'], []]


# 2d. Stemming
## Stemming reduces words to their word stems (e.g., assaulted -> assault)

In [15]:
#Import NLTK's SnowballStemmer

from nltk.stem import SnowballStemmer 
stemmer = SnowballStemmer("english")
def stemming(texts):
    texts_out = []
    for token in texts:
        texts_out.append([stemmer.stem(t) for t in token])
    return texts_out

In [48]:
##Stem Full dataset

GI_Author_merged_stemmed = stemming(GI_Author_merged_stopwords)


print("\n[INFO] GI_Author_merged....................\n")
print(GI_Author_merged_stemmed[:2])





[INFO] GI_Author_merged....................

[['get', 'expos', 'sexual', 'violenc', 'report', 'unbias', 'manner', 'get', 'interest', 'stori', 'pass', 'class', 'rais', 'awar', 'report', 'fact', 'get', 'engag', 'write', 'someth', 'peopl', 'care', 'expos', 'guy'], []]


# 2e. Bigrams and Trigrams
##    Bigrams are two words that frequently co-occur together
##    Trigrams are three words that frequently co-occur together

In [49]:
#Trigrams and Bigrams in full dataset
            
GI_Author_merged_bigram = Phrases(GI_Author_merged_stemmed, min_count=3, delimiter=b' ', threshold=1)
GI_Author_merged_trigram = Phrases(GI_Author_merged_bigram[GI_Author_merged_stemmed], threshold=1)

GI_Author_merged_bigram_mod = gensim.models.phrases.Phraser(GI_Author_merged_bigram)
GI_Author_merged_trigram_mod = gensim.models.phrases.Phraser(GI_Author_merged_trigram)

for idx in range(len(GI_Author_merged_stemmed)):
    for token in GI_Author_merged_trigram_mod[GI_Author_merged_bigram_mod[GI_Author_merged_stemmed[idx]]]:
        #print(token)
        if ' ' in token:
            GI_Author_merged_stemmed[idx].append(token)
print("\n[INFO] GI_Author_merged....................\n")
print(GI_Author_merged_stemmed[:2])


[INFO] GI_Author_merged....................

[['get', 'expos', 'sexual', 'violenc', 'report', 'unbias', 'manner', 'get', 'interest', 'stori', 'pass', 'class', 'rais', 'awar', 'report', 'fact', 'get', 'engag', 'write', 'someth', 'peopl', 'care', 'expos', 'guy', 'rais awar', 'peopl care', 'expos guy'], []]


# 2f. Remove words that occur in less than & greater than of documents
### The corpus is our collection of documents (i.e., our textual questionnaire responses)
### The dictionary takes each unique word in the corpus and assigns them an index

In [50]:
## Generate Corpus and Dictionary for full dataset

dictionary_GI_Author_merged = corpora.Dictionary(GI_Author_merged_stemmed)
dictionary_GI_Author_merged.filter_extremes(no_below=.01, no_above=0.99)
corpus_GI_Author_merged = [dictionary_GI_Author_merged.doc2bow(text) for text in GI_Author_merged_stemmed]
pickle.dump(corpus_GI_Author_merged, open('/Users/hannahstevens/Desktop/LDA_Main/project2/data_new/corpus_GI_Author_merged.pkl', 'wb'))
dictionary_GI_Author_merged.save('/Users/hannahstevens/Desktop/LDA_Main/project2/data_new/dictionary_GI_Author_merged.gensim')

# 3. Model Selection- Selecting the number of topics (k)

## 3a. Setting Model Hyperparameters 

### 1. Beta (referred to as 'eta' in gensim) = the [distribution of the] number of words per topic
### 2. Alpha =  the [distribution of the] number of topics per document

#### Both alpha and eta can be set to ‘symmetric’, ‘asymmetric’, or ‘auto’:
        - ‘auto’ = the model learns the best values for the hyperparameters as it is trained       
                   on more and more data (i.e., it learns an asymmetric prior from the corpus). See 
                   http://jonathan-huang.org/research/dirichlet/dirichlet.pdf for an overview             
        - 'asymmetric' = uses a fixed, normalized asymmetric prior of 1.0 / k (number of topics)
        - 'symmetric' = uses a distribution of 1 / k (number of topics)

In Bayesian statistics, we have to define the distributions (i.e., prior distributions) of unknown variables (e.g., ϕ and θ) before running the data analysis. These should be defined based on theoretical assumptions about how we think the topics are actually distributed amongst our data. In our case, it makes sense to assume that some documents discuss more/less topics than other documents; thus we set the document-topic distribution to be asymmetric. 
#### Thus, we reccommend setting alpha = 'auto' as it sets the distribution to be asymmetric, and learns the best alpha value (i.e., lowest perplexity scores) from the data itself. It also makes sense to assume that some topics contain more words than others. Thus, we reccomend setting the distribution of the number of words per topic to be asymmetric as well. 

### 3. Passes = number of laps the model goes through the entire corpus
        - Incrasing the number of passes reduces model bias
### 4. Chunksize = number of documents to load into memory at a time 
        - smaller chunksizes save memory, but take longer to train
### 5. Update_every = number of chunks to process before maximizing your model 
### 6. Random state = sets the seed to make the model reproducable
### 7. Number of topics (k)
Researchers must tell the model how many (k) prominent goal inference topics to sort each ‘bag of words’ document into. Problematically, several different k-values might work. Thus, we use a metric called perplexity to help us to determine the optimal number of topics. The utility in perplexity comes from comparing perplexity values across models with differing k-values to pinpoint the best model (i.e., the model with the lowest perplexity score). 

Thus, we recommend testing the perplexity of the model with a variety of k values, and then runing the final model using the k-value with the lowest perplexity score.


In [51]:
## Model Hyper Parameters
k = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32]
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta= 'auto'
per_word_topics=True

lda_model_GI_Author_merged = []

## 3b. Compute model perplexity scores
We run the model with different topic numbers (k) to find the optimal topics number

We will start by looking at k=1-32 topics

In [53]:
#Get Perplexity Scores of Training Dataset

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")


for i in k:

    lda_model_GI_Author_merged = gensim.models.ldamodel.LdaModel(corpus=corpus_GI_Author_merged,
                                                  id2word=dictionary_GI_Author_merged,
                                                  num_topics=i, 
                                                  random_state=random_state,
                                                  update_every=update_every,
                                                  chunksize=chunksize,
                                                  passes=passes,
                                                  iterations=iterations,
                                                  alpha=alpha,
                                                  eta=eta,                                                            
                                                  per_word_topics=per_word_topics)

    print('\nPerplexity (num_topics = {}): '.format(i), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (num_topics = 1):  -6.137480737839619

Perplexity (num_topics = 2):  -6.19238248944063

Perplexity (num_topics = 3):  -6.22206057045652

Perplexity (num_topics = 4):  -6.278549368621163

Perplexity (num_topics = 5):  -6.308312053544437

Perplexity (num_topics = 6):  -6.349260912652795

Perplexity (num_topics = 7):  -6.398540728499678

Perplexity (num_topics = 8):  -6.405206289114514

Perplexity (num_topics = 9):  -6.4497211203536216

Perplexity (num_topics = 10):  -6.454788288375014

Perplexity (num_topics = 11):  -6.474283834833145

Perplexity (num_topics = 12):  -6.4896297378138295

Perplexity (num_topics = 13):  -6.487679668905522

Perplexity (num_topics = 14):  -6.514942912623576

Perplexity (num_topics = 15):  -6.508571582672518

Perplexity (num_topics = 16):  -6.53059214854

# 3c. Model selection using pyLDAvis Visualization 
## We're interested in the models with k= 3,4,5,6,7 topics, so we visualize those using the pyLDAvis documentation (https://www.aclweb.org/anthology/W14-3110.pdf)

## When selecting the optimal number of topics, we need to find a balance between underfitting and underfitting the model

### OVERFITTING (i.e., too many topics): 
#### Practical takeaway- this can make it harder for human coders to label
#### pros- less overlap amongst topics
#### cons- less coherence amongst the words in each topic; decreased varaiance in each document's distirbution of topics

### UNDERFITTING (i.e., too few topics): 
#### Practical takeaway- doesn't produce enough variance, limiting options for statistical analyses
#### pros- more coherent 'bag of words' comprising each topic; increased varaiance in the distirbution of topics in each document
#### cons- more overlap amongst topics

In [79]:
# Initializing LDA Models and Parameters
topic_number = 3
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

# Full Dataset

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")


lda_model_GI_Author_merged = gensim.models.ldamodel.LdaModel(corpus=corpus_GI_Author_merged,
                                                  id2word=dictionary_GI_Author_merged,
                                                  num_topics=topic_number, 
                                                  random_state=random_state,
                                                  update_every=update_every,
                                                  chunksize=chunksize,
                                                  passes=passes,
                                                  iterations=iterations,
                                                  alpha=alpha,
                                                  eta=eta,
                                                  per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (topic_number = 3):  -6.22206057045652


## Reading pyLDAvis

### LEFT PANE:
- The area of each circle represents the prevalence of each topic over the entire corpus 
- The distance between the center of circles indicate the similarity between topics (i.e., inter-topic differences)

---------------------------------------------------------------------------------------------------------

### RIGHT PANE:
- If you hover over a particular topic on the left, the histogram on the right side lists the top 30 most relevant terms
- The widths of the gray bars represent the corpus-wide frequencies of each term, and the widths of the red bars represent the topic-specific frequencies of each term
- A slider at the top can adjust the relevence metric (λ); however, for our purposes, be sure it i set to λ = 1. For more information on the relevance metric, see (https://www.aclweb.org/anthology/W14-3110.pdf). 








### k = 3 topics

In [80]:

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 3...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)


***********************************************************************
[INFO] GI_Author_Merged Full Dataset Model Results....
***********************************************************************

[INFO] Num_topics: 3

(0, '0.026*"inform" + 0.018*"peopl" + 0.016*"give" + 0.016*"happen" + 0.016*"male" + 0.016*"stori" + 0.015*"incid" + 0.013*"make" + 0.013*"tri" + 0.013*"situat"')
(1, '0.040*"sexual" + 0.037*"assault" + 0.029*"show" + 0.025*"sexual assault" + 0.021*"male" + 0.017*"student" + 0.016*"peopl" + 0.015*"awar" + 0.014*"tri" + 0.014*"guy"')
(2, '0.035*"make" + 0.032*"male" + 0.019*"accus" + 0.019*"stori" + 0.018*"side" + 0.017*"guy" + 0.015*"like" + 0.014*"tri" + 0.014*"seem" + 0.013*"girl"')
GI_Author_Merged.....k = 3...................


In [81]:
# Initializing LDA Models and Parameters
topic_number = 4
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

# Full Dataset

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")


lda_model_GI_Author_merged = gensim.models.ldamodel.LdaModel(corpus=corpus_GI_Author_merged,
                                                  id2word=dictionary_GI_Author_merged,
                                                  num_topics=topic_number, 
                                                  random_state=random_state,
                                                  update_every=update_every,
                                                  chunksize=chunksize,
                                                  passes=passes,
                                                  iterations=iterations,
                                                  alpha=alpha,
                                                  eta=eta,
                                                  per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (topic_number = 4):  -6.278549368621163


### k = 4 topics

In [82]:

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 4...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)


***********************************************************************
[INFO] GI_Author_Merged Full Dataset Model Results....
***********************************************************************

[INFO] Num_topics: 4

(0, '0.032*"make" + 0.023*"peopl" + 0.022*"male" + 0.016*"happen" + 0.016*"tri" + 0.015*"inform" + 0.015*"incid" + 0.013*"want" + 0.012*"awar" + 0.011*"stori"')
(1, '0.045*"sexual" + 0.042*"assault" + 0.029*"show" + 0.027*"sexual assault" + 0.026*"male" + 0.019*"student" + 0.014*"stori" + 0.013*"guy" + 0.013*"parti" + 0.013*"peopl"')
(2, '0.028*"stori" + 0.024*"accus" + 0.021*"side" + 0.020*"tell" + 0.015*"happen" + 0.014*"describ" + 0.012*"event" + 0.011*"paint" + 0.009*"girl" + 0.009*"tri"')
(3, '0.030*"inform" + 0.022*"public" + 0.021*"male" + 0.021*"accus" + 0.019*"guy" + 0.017*"make" + 0.017*"situat" + 0.016*"tri" + 0.016*"rape" + 0.012*"inform public"')
GI_Author_Merged.....k = 4...................


In [83]:
# Initializing LDA Models and Parameters
topic_number = 5
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

# Full Dataset

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")


lda_model_GI_Author_merged = gensim.models.ldamodel.LdaModel(corpus=corpus_GI_Author_merged,
                                                  id2word=dictionary_GI_Author_merged,
                                                  num_topics=topic_number, 
                                                  random_state=random_state,
                                                  update_every=update_every,
                                                  chunksize=chunksize,
                                                  passes=passes,
                                                  iterations=iterations,
                                                  alpha=alpha,
                                                  eta=eta,
                                                  per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (topic_number = 5):  -6.308312053544437


### k = 5 topics

In [84]:

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 5...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)


***********************************************************************
[INFO] GI_Author_Merged Full Dataset Model Results....
***********************************************************************

[INFO] Num_topics: 5

(0, '0.035*"make" + 0.027*"male" + 0.022*"peopl" + 0.017*"tri" + 0.015*"happen" + 0.014*"awar" + 0.013*"incid" + 0.012*"like" + 0.012*"inform" + 0.012*"guy"')
(1, '0.042*"sexual" + 0.040*"assault" + 0.032*"show" + 0.026*"male" + 0.025*"sexual assault" + 0.019*"student" + 0.015*"parti" + 0.015*"happen" + 0.015*"stori" + 0.014*"guy"')
(2, '0.034*"tell" + 0.023*"stori" + 0.020*"accus" + 0.017*"peopl" + 0.015*"tri" + 0.014*"event" + 0.013*"side" + 0.013*"make" + 0.013*"happen" + 0.011*"tell peopl"')
(3, '0.033*"inform" + 0.021*"public" + 0.019*"male" + 0.017*"sexual" + 0.017*"situat" + 0.016*"guy" + 0.015*"assault" + 0.015*"rape" + 0.014*"accus" + 0.013*"make"')
(4, '0.021*"accus" + 0.020*"stori" + 0.020*"student" + 0.018*"tri" + 0.015*"male" + 0.014*"inform" + 0.014*"sh

In [85]:
# Initializing LDA Models and Parameters
topic_number = 6
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

# Full Dataset

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")


lda_model_GI_Author_merged = gensim.models.ldamodel.LdaModel(corpus=corpus_GI_Author_merged,
                                                  id2word=dictionary_GI_Author_merged,
                                                  num_topics=topic_number, 
                                                  random_state=random_state,
                                                  update_every=update_every,
                                                  chunksize=chunksize,
                                                  passes=passes,
                                                  iterations=iterations,
                                                  alpha=alpha,
                                                  eta=eta,
                                                  per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (topic_number = 6):  -6.349260912652795


In [86]:

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 6...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)


***********************************************************************
[INFO] GI_Author_Merged Full Dataset Model Results....
***********************************************************************

[INFO] Num_topics: 6

(0, '0.030*"peopl" + 0.021*"happen" + 0.018*"stori" + 0.017*"incid" + 0.017*"tri" + 0.016*"inform" + 0.016*"male" + 0.015*"awar" + 0.015*"side" + 0.013*"give"')
(1, '0.042*"show" + 0.036*"sexual" + 0.034*"assault" + 0.031*"male" + 0.019*"sexual assault" + 0.018*"tri" + 0.017*"student" + 0.016*"peopl" + 0.016*"guy" + 0.014*"happen"')
(2, '0.051*"tell" + 0.040*"stori" + 0.025*"tell stori" + 0.016*"happen" + 0.016*"side" + 0.013*"event" + 0.013*"describ" + 0.013*"tri" + 0.012*"tell peopl" + 0.012*"peopl"')
(3, '0.043*"inform" + 0.037*"sexual" + 0.034*"assault" + 0.029*"public" + 0.023*"sexual assault" + 0.019*"awar" + 0.017*"situat" + 0.016*"male" + 0.016*"inform public" + 0.013*"accus"')
(4, '0.027*"give" + 0.025*"stori" + 0.025*"student" + 0.019*"awar" + 0.016*"side" 

# k=6 topics looks the best to me! The topics appear to be relatively spread out, with no overlapping topics
# At the same time, the 'bag of words' comprising each topic appears coherent enough to label. 

In [87]:
# Initializing LDA Models and Parameters
topic_number = 7
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

# Full Dataset

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")


lda_model_GI_Author_merged = gensim.models.ldamodel.LdaModel(corpus=corpus_GI_Author_merged,
                                                  id2word=dictionary_GI_Author_merged,
                                                  num_topics=topic_number, 
                                                  random_state=random_state,
                                                  update_every=update_every,
                                                  chunksize=chunksize,
                                                  passes=passes,
                                                  iterations=iterations,
                                                  alpha=alpha,
                                                  eta=eta,
                                                  per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (topic_number = 7):  -6.398540728499678


### k = 7 topics

In [88]:

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 7...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)


***********************************************************************
[INFO] GI_Author_Merged Full Dataset Model Results....
***********************************************************************

[INFO] Num_topics: 7

(0, '0.027*"peopl" + 0.024*"inform" + 0.022*"happen" + 0.022*"give" + 0.019*"male" + 0.018*"awar" + 0.018*"incid" + 0.016*"tri" + 0.014*"situat" + 0.012*"unbias"')
(1, '0.053*"sexual" + 0.050*"assault" + 0.041*"show" + 0.033*"sexual assault" + 0.023*"male" + 0.018*"peopl" + 0.018*"guy" + 0.017*"report" + 0.016*"happen" + 0.015*"parti"')
(2, '0.058*"tell" + 0.045*"stori" + 0.028*"tell stori" + 0.019*"happen" + 0.018*"side" + 0.018*"event" + 0.017*"describ" + 0.015*"peopl" + 0.012*"tell peopl" + 0.012*"girl"')
(3, '0.045*"inform" + 0.044*"public" + 0.023*"inform public" + 0.022*"assault" + 0.021*"sexual" + 0.021*"situat" + 0.015*"accus" + 0.015*"provid" + 0.015*"sexual assault" + 0.015*"awar"')
(4, '0.046*"side" + 0.046*"stori" + 0.033*"give" + 0.024*"inform" + 0.021*"

In [89]:
# Initializing LDA Models and Parameters
topic_number = 8
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

# Full Dataset

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")


lda_model_GI_Author_merged = gensim.models.ldamodel.LdaModel(corpus=corpus_GI_Author_merged,
                                                  id2word=dictionary_GI_Author_merged,
                                                  num_topics=topic_number, 
                                                  random_state=random_state,
                                                  update_every=update_every,
                                                  chunksize=chunksize,
                                                  passes=passes,
                                                  iterations=iterations,
                                                  alpha=alpha,
                                                  eta=eta,
                                                  per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (topic_number = 8):  -6.405206289114514


### k = 8 topics

In [90]:

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 8...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)


***********************************************************************
[INFO] GI_Author_Merged Full Dataset Model Results....
***********************************************************************

[INFO] Num_topics: 8

(0, '0.038*"peopl" + 0.030*"awar" + 0.023*"inform" + 0.023*"happen" + 0.019*"incid" + 0.018*"tri" + 0.016*"make" + 0.015*"know" + 0.015*"situat" + 0.014*"get"')
(1, '0.043*"show" + 0.025*"male" + 0.025*"assault" + 0.022*"sexual" + 0.020*"guy" + 0.019*"report" + 0.019*"tri" + 0.017*"student" + 0.016*"peopl" + 0.015*"stori"')
(2, '0.023*"stori" + 0.023*"tell" + 0.022*"side" + 0.022*"accus" + 0.019*"happen" + 0.015*"describ" + 0.015*"light" + 0.014*"paint" + 0.013*"event" + 0.012*"girl"')
(3, '0.056*"public" + 0.048*"inform" + 0.032*"inform public" + 0.025*"situat" + 0.025*"male" + 0.021*"rape" + 0.020*"provid" + 0.014*"make" + 0.014*"stori" + 0.013*"victim"')
(4, '0.025*"accus" + 0.022*"stori" + 0.022*"inform" + 0.017*"side" + 0.017*"give" + 0.016*"show" + 0.015*"tri" 

In [91]:
# Initializing LDA Models and Parameters
topic_number = 9
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

# Full Dataset

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")


lda_model_GI_Author_merged = gensim.models.ldamodel.LdaModel(corpus=corpus_GI_Author_merged,
                                                  id2word=dictionary_GI_Author_merged,
                                                  num_topics=topic_number, 
                                                  random_state=random_state,
                                                  update_every=update_every,
                                                  chunksize=chunksize,
                                                  passes=passes,
                                                  iterations=iterations,
                                                  alpha=alpha,
                                                  eta=eta,
                                                  per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (topic_number = 9):  -6.4497211203536216


### k = 9 topics

In [92]:
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 9...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)


***********************************************************************
[INFO] GI_Author_Merged Full Dataset Model Results....
***********************************************************************

[INFO] Num_topics: 9

(0, '0.057*"peopl" + 0.034*"awar" + 0.032*"happen" + 0.028*"inform" + 0.024*"incid" + 0.023*"know" + 0.023*"tri" + 0.018*"parti" + 0.017*"get" + 0.017*"let"')
(1, '0.034*"show" + 0.026*"peopl" + 0.025*"sexual" + 0.023*"male" + 0.022*"guy" + 0.022*"assault" + 0.019*"report" + 0.018*"warn" + 0.017*"tri" + 0.016*"want"')
(2, '0.072*"stori" + 0.053*"tell" + 0.033*"side" + 0.030*"tell stori" + 0.021*"describ" + 0.020*"event" + 0.020*"tri" + 0.017*"happen" + 0.015*"object" + 0.015*"think"')
(3, '0.070*"public" + 0.062*"inform" + 0.037*"inform public" + 0.021*"situat" + 0.021*"accus" + 0.015*"provid" + 0.015*"background" + 0.013*"make" + 0.013*"rape" + 0.012*"male"')
(4, '0.043*"give" + 0.030*"side" + 0.029*"stori" + 0.023*"show" + 0.018*"inform" + 0.018*"accus" + 0.018*"tr

## 5d. Run Final Model

In [93]:
# Initializing LDA Models and Parameters
topic_number = 6
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

# Full Dataset

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")


lda_model_GI_Author_merged = gensim.models.ldamodel.LdaModel(corpus=corpus_GI_Author_merged,
                                                  id2word=dictionary_GI_Author_merged,
                                                  num_topics=topic_number, 
                                                  random_state=random_state,
                                                  update_every=update_every,
                                                  chunksize=chunksize,
                                                  passes=passes,
                                                  iterations=iterations,
                                                  alpha=alpha,
                                                  eta=eta,
                                                  per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (topic_number = 6):  -6.349260912652795


In [106]:
#GI_Author_Merged Model Results

print("\n***********************************************************************")
print("[INFO] Goal Inferences Model Results....")
print("***********************************************************************")

print("\n[INFO] Number of topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("Goal Inferences .....k = 6...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)


***********************************************************************
[INFO] Goal Inferences Model Results....
***********************************************************************

[INFO] Number of topics: 6

(0, '0.030*"peopl" + 0.021*"happen" + 0.018*"stori" + 0.017*"incid" + 0.017*"tri" + 0.016*"inform" + 0.016*"male" + 0.015*"awar" + 0.015*"side" + 0.013*"give"')
(1, '0.042*"show" + 0.036*"sexual" + 0.034*"assault" + 0.031*"male" + 0.019*"sexual assault" + 0.018*"tri" + 0.017*"student" + 0.016*"peopl" + 0.016*"guy" + 0.014*"happen"')
(2, '0.051*"tell" + 0.040*"stori" + 0.025*"tell stori" + 0.016*"happen" + 0.016*"side" + 0.013*"event" + 0.013*"describ" + 0.013*"tri" + 0.012*"tell peopl" + 0.012*"peopl"')
(3, '0.043*"inform" + 0.037*"sexual" + 0.034*"assault" + 0.029*"public" + 0.023*"sexual assault" + 0.019*"awar" + 0.017*"situat" + 0.016*"male" + 0.016*"inform public" + 0.013*"accus"')
(4, '0.027*"give" + 0.025*"stori" + 0.025*"student" + 0.019*"awar" + 0.016*"side" + 0.014*

## 5. Save the analysis results to an excel file for topic validation

### 5a. Here we generate a column that tells us which topic each response contributed the most to

In [100]:
cols = [color for name, color in mcolors.XKCD_COLORS.items()]
mycolors = [color for name, color in mcolors.XKCD_COLORS.items()]

In [101]:
def format_topics_sentences(ldamodel, corpus, texts):
    # Init output
    sent_topics_df = pd.DataFrame()

    # Get the main topic of each document
    for i, row_list in enumerate(ldamodel[corpus]):
        row = row_list[0] if ldamodel.per_word_topics else row_list            
        # print(row)
        row = sorted(row, key=lambda x: (x[1]), reverse=True)
        # Get the Dominant topic, Perc Contribution, and Keywords for each document
        raw_frame = {}
        for j, (topic_num, prop_topic) in enumerate(row):
            #if j < 2:  # => dominant topic
                #wp = ldamodel.show_topic(topic_num)
                #topic_keywords = ", ".join([word for word, prop in wp])

                #sent_topics_df = sent_topics_df.append(pd.Series([int(topic_num), round(prop_topic,7), topic_keywords]), ignore_index=True)
            if j==0:
                raw_frame['Dominant'] = topic_num

            raw_frame['Topic' + str(topic_num)] = round(prop_topic, 4)

            #else:
            #    break
        df = pd.DataFrame(data=raw_frame, index=[0])
        sent_topics_df = sent_topics_df.append(df)
        
    #sent_topics_df.columns = ['Dominant_Topic', 'Perc_Contribution', 'Topic_Keywords']
    #sent_topics_df.columns = ['Dominant_Topic', 'Perc_Contribution']

    # Add original text to the end of the output
    #contents = pd.Series(texts)
    #sent_topics_df = pd.concat([sent_topics_df, contents], axis=1)
    return(sent_topics_df)


df_topic_sents_keywords_GI_Author_merged = format_topics_sentences(ldamodel=lda_model_GI_Author_merged, corpus=corpus_GI_Author_merged, texts=GI_Author_merged_stopwords)

df_dominant_topic_GI_Author_merged = df_topic_sents_keywords_GI_Author_merged.reset_index()
# Format
df_dominant_topic_GI_Author_merged.index.name='Document_No';

print(df_dominant_topic_GI_Author_merged.head(812))
# Format
#df_dominant_topic_GI_Author_merged_train = df_topic_sents_keywords_GI_Author_merged_train.reset_index()
#df_dominant_topic_GI_Author_merged_train.columns = ['Document_No', 'Dominant_Topic', 'Topic_Perc_Contrib', 'Keywords', 'Text']
#df_dominant_topic_GI_Author_merged_train.head(3)

             index  Dominant  Topic0  Topic1  Topic2  Topic3  Topic4  Topic5
Document_No                                                                 
0                0         0  0.7163  0.2794     NaN     NaN     NaN     NaN
1                0         1  0.1935  0.2431  0.0638  0.1823  0.0810  0.2363
2                0         1  0.1935  0.2431  0.0638  0.1823  0.0810  0.2363
3                0         1  0.1935  0.2431  0.0638  0.1823  0.0810  0.2363
4                0         1  0.1935  0.2431  0.0638  0.1823  0.0810  0.2363
5                0         1     NaN  0.5797     NaN  0.4138     NaN     NaN
6                0         1     NaN  0.9881     NaN     NaN     NaN     NaN
7                0         5     NaN     NaN     NaN  0.4812     NaN  0.5057
8                0         1  0.1114  0.6315     NaN  0.2525     NaN     NaN
9                0         0  0.6759     NaN     NaN  0.3135     NaN     NaN
10               0         5     NaN     NaN     NaN     NaN     NaN  0.9870

## 5b. Generate a dataframe to export the results into

In [102]:
lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Dominant'])
topic0_contrib_lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Topic0'])
topic1_contrib_lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Topic1'])
topic2_contrib_lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Topic2'])
topic3_contrib_lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Topic3'])
topic4_contrib_lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Topic4'])
topic5_contrib_lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Topic5'])

GI_Author_merged = np.array(data['GI_Author_merged'])

ResponseId = np.array(data['ResponseId'])
uid = np.array(data['uid'])

results = { 
    'uid' : uid,'ResponseId': ResponseId, 
    'GI_Author_merged': GI_Author_merged, 
    'lda_topics_GI_Author_merged': lda_topics_GI_Author_merged, 
    'topic0_contrib_lda_topics_GI_Author_merged':topic0_contrib_lda_topics_GI_Author_merged,
    'topic1_contrib_lda_topics_GI_Author_merged':topic1_contrib_lda_topics_GI_Author_merged,
    'topic2_contrib_lda_topics_GI_Author_merged':topic2_contrib_lda_topics_GI_Author_merged,
    'topic3_contrib_lda_topics_GI_Author_merged':topic3_contrib_lda_topics_GI_Author_merged,
    'topic4_contrib_lda_topics_GI_Author_merged':topic4_contrib_lda_topics_GI_Author_merged,
    'topic5_contrib_lda_topics_GI_Author_merged':topic5_contrib_lda_topics_GI_Author_merged,

}

frame = pd.DataFrame(results, columns = [
                                                'uid', 'ResponseId',
                                                'GI_Author_merged', 'lda_topics_GI_Author_merged', 
                                                'topic0_contrib_lda_topics_GI_Author_merged',
                                                'topic1_contrib_lda_topics_GI_Author_merged',
                                                'topic2_contrib_lda_topics_GI_Author_merged',
                                                'topic3_contrib_lda_topics_GI_Author_merged',
                                                'topic4_contrib_lda_topics_GI_Author_merged',
                                                'topic5_contrib_lda_topics_GI_Author_merged',

                                              ])



frame.to_excel("/Users/hannahstevens/Desktop/LDA_Main/project2/data_new/lda_results_full_dataset_topic_num_5.xlsx")

## 5c. Export restuls to an .xlsx file

In [103]:
'/Users/hannahstevens/Desktop/LDA_Main/project2/data_new/experimental_data.xlsx'
frame.to_excel("/Users/hannahstevens/Desktop/LDA_Main/project2/data_new/lda_results_All_data_topic_num_5.xlsx")